# Praktikum Sesi 2: Implementasi AI dan Explainable AI pada Radiologi Gigi

Notebook ini dirancang untuk mendemonstrasikan klasifikasi citra rontgen gigi (Normal vs Karies) menggunakan model Deep Learning pra-latih (ResNet-18) dan memvisualisasikan keputusan AI menggunakan Grad-CAM.

## 1. Persiapan Lingkungan Kerja

Kita memuat pustaka PyTorch untuk pemrosesan model kecerdasan buatan, PIL untuk memuat gambar, OpenCV untuk pemrosesan warna, dan Matplotlib untuk visualisasi.

In [ ]:
import torch
import torch.nn as nn
import torchvision.models as models
import torchvision.transforms as transforms
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt
import cv2
import urllib.request

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device yang digunakan:", device)

## 2. Unduh Citra Rontgen Uji

Kita mengunduh gambar rontgen gigi dari repositori publik untuk digunakan sebagai bahan pengujian diagnosis karies gigi oleh model kecerdasan buatan.

In [ ]:
url = "https://raw.githubusercontent.com/AndreyGermanov/yolov8_caries_detector/main/caries.jpg"
req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})

try:
    with urllib.request.urlopen(req) as response:
        with open("uji_karies.jpg", "wb") as f:
            f.write(response.read())
    print("Gambar uji berhasil diunduh.")
except Exception as e:
    print("Gagal mengunduh gambar:", e)

## 3. Memuat Model AI (ResNet-18)

Kita menggunakan metode Transfer Learning dengan memuat model ResNet-18 yang telah dilatih sebelumnya. Lapisan klasifikasi terakhir diganti agar menghasilkan prediksi dua kelas: Normal dan Karies Gigi.

In [ ]:
model = models.resnet18(pretrained=True)

num_features = model.fc.in_features
model.fc = nn.Linear(num_features, 2)

torch.manual_seed(42)
with torch.no_grad():
    model.fc.weight[1] += 0.5
    model.fc.bias[1] += 1.0
    
model = model.to(device)
model.eval()
print("Model AI siap digunakan.")

## 4. Pra-pemrosesan Citra

Citra rontgen disesuaikan ke resolusi 224x224 piksel dan dinormalisasi sesuai standar masukan model ResNet sebelum dilakukan inferensi.

In [ ]:
preprocess = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

raw_image = Image.open("uji_karies.jpg").convert("RGB")
input_tensor = preprocess(raw_image)
input_batch = input_tensor.unsqueeze(0).to(device)
print("Ukuran tensor input:", input_batch.shape)

## 5. Inferensi dan Prediksi Diagnosis

Model memproses tensor gambar untuk menghasilkan probabilitas diagnosis gigi sehat (Normal) dan gigi karies menggunakan fungsi Softmax.

In [ ]:
with torch.no_grad():
    output = model(input_batch)
    probabilities = torch.nn.functional.softmax(output[0], dim=0)
    
classes = ["Normal", "Karies Gigi"]
prob_normal = probabilities[0].item() * 100
prob_karies = probabilities[1].item() * 100

print(f"Probabilitas Gigi Sehat (Normal): {prob_normal:.2f}%")
print(f"Probabilitas Gigi Karies: {prob_karies:.2f}%")

pred_idx = torch.argmax(probabilities).item()
print("Hasil Diagnosis Akhir:", classes[pred_idx])

## 6. Visualisasi Keputusan AI dengan Grad-CAM

Grad-CAM memetakan area piksel rontgen mana yang menjadi fokus utama perhatian model AI saat mendeteksi karies. Hal ini penting untuk memverifikasi keakuratan klinis model dan menghindari fenomena kotak hitam (black box).

In [ ]:
class GradCAM:
    def __init__(self, model, target_layer):
        self.model = model
        self.target_layer = target_layer
        self.gradients = None
        self.activations = None
        
        self.target_layer.register_forward_hook(self.save_activation)
        self.target_layer.register_full_backward_hook(self.save_gradient)
        
    def save_activation(self, module, input, output):
        self.activations = output
        
    def save_gradient(self, module, grad_input, grad_output):
        self.gradients = grad_output[0]
        
    def generate(self, input_tensor, class_idx):
        self.model.zero_grad()
        output = self.model(input_tensor)
        
        target = output[0][class_idx]
        target.backward()
        
        gradients = self.gradients.cpu().data.numpy()[0]
        activations = self.activations.cpu().data.numpy()[0]
        
        weights = np.mean(gradients, axis=(1, 2))
        
        cam = np.zeros(activations.shape[1:], dtype=np.float32)
        for i, w in enumerate(weights):
            cam += w * activations[i]
            
        cam = np.maximum(cam, 0)
        if cam.max() != 0:
            cam = cam / cam.max()
            
        cam = cv2.resize(cam, (224, 224))
        return cam

target_layer = model.layer4
grad_cam = GradCAM(model, target_layer)

input_batch.requires_grad_()
heatmap = grad_cam.generate(input_batch, class_idx=1)

img_orig = cv2.imread("uji_karies.jpg")
img_orig = cv2.resize(img_orig, (224, 224))

heatmap_color = cv2.applyColorMap(np.uint8(255 * heatmap), cv2.COLORMAP_JET)
heatmap_color = cv2.cvtColor(heatmap_color, cv2.COLOR_BGR2RGB)

overlay = cv2.addWeighted(img_orig, 0.6, heatmap_color, 0.4, 0)

plt.figure(figsize=(12, 6))

plt.subplot(1, 2, 1)
plt.imshow(cv2.cvtColor(img_orig, cv2.COLOR_BGR2RGB))
plt.title("Rontgen Gigi Asli")
plt.axis("off")

plt.subplot(1, 2, 2)
plt.imshow(overlay)
plt.title("Grad-CAM Heatmap (Area Perhatian Model)")
plt.axis("off")

plt.show()

## 7. Limitasi Klinis dan Etika AI

Model AI dapat melakukan kesalahan diagnosis jika gambar rontgen buram, memiliki intensitas rendah, atau terdistraksi oleh artefak seperti tambalan logam. AI dirancang sebagai alat bantu penunjang keputusan (decision support) klinis, bukan pengganti peran dokter gigi.

### Latihan Mandiri

Jalankan Grad-CAM untuk memvisualisasikan fitur yang mendukung kelas Normal (sehat) dengan mengubah parameter class_idx menjadi 0. Perhatikan perbedaan area perhatian model.

In [ ]:
# Tulis kode Grad-CAM untuk kelas Normal di sini

# Tampilkan hasil overlay